In [10]:
# Accuracy-precision plots and pareto curve
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import font_manager


# =========================
# 0. Font setting
# =========================
font_manager.fontManager.addfont("Times New Roman.ttf")
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 16
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14
plt.rcParams["legend.fontsize"] = 12


# =========================
# 1. Path setting
# =========================
csv_path = "tree_pareto_blsc_results.csv"
out_dir = "tree_pareto_curve"

methods_single = ['CART', 'U-BinOCT', 'BendersOCT', 'U-PIP']
methods_pareto = ['C-PIP', 'FlowOCT', 'C-BinOCT']

method_order = ['C-PIP', 'FlowOCT', 'C-BinOCT', 'U-PIP', 'BendersOCT', 'U-BinOCT', 'CART']

METHOD_STYLE = {
    'C-PIP':      {'color': 'blue',      'marker': 'o', 'size': 40},
    'FlowOCT':    {'color': 'purple',    'marker': '^', 'size': 40},
    'C-BinOCT':   {'color': 'deeppink',  'marker': 'v', 'size': 40},
    'U-PIP':      {'color': '#FF8000',    'marker': 's', 'size': 80},
    'BendersOCT': {'color': 'green',     'marker': 'D', 'size': 70},
    'U-BinOCT':   {'color': '#CC0000',       'marker': 'P', 'size': 90},
    'CART':       {'color': '#009999',   'marker': 'X', 'size': 90},
}

# Pareto linestyle
METHOD_LINESTYLE = {
    'C-PIP': '-',
    'FlowOCT': '--',
    'C-BinOCT': '-.',
}

method_color_map = {m: METHOD_STYLE[m]['color'] for m in METHOD_STYLE}
method_marker_map = {m: METHOD_STYLE[m]['marker'] for m in METHOD_STYLE}
method_size_map = {m: METHOD_STYLE[m]['size'] for m in METHOD_STYLE}

# Coordinate range
xlim_left, xlim_right = 0.0, 1.0
ylim_left, ylim_right = 0.0, 1.0
tick_values = np.arange(0.0, 1.01, 0.1)
SAVE_PAD_INCHES = 0.20

modes = ["train", "test"]   
n_rows = 2
n_cols = 4                  


# =========================
# 2. Pareto no dominated point
# =========================
def pareto_frontier_max(df_xy: pd.DataFrame, x: str, y: str) -> pd.DataFrame:
    if df_xy.empty:
        return df_xy.copy()

    tmp = df_xy[[x, y]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if tmp.empty:
        return tmp

    X = tmp[x].to_numpy()
    Y = tmp[y].to_numpy()
    n = len(tmp)

    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]:
            continue
        dominated = ((X > X[i]) & (Y >= Y[i])) | ((X >= X[i]) & (Y > Y[i]))
        if dominated.any():
            keep[i] = False

    pareto = tmp.loc[keep].copy()
    pareto = pareto.sort_values(by=[x, y], ascending=[True, True]).reset_index(drop=True)
    return pareto


def plot_staircase_pareto(ax, pareto_df, x_col, y_col,
                          xlim_left, xlim_right, ylim_left, ylim_right,
                          color, linestyle='-'):
    if pareto_df.empty:
        return

    x0, y0 = pareto_df.iloc[0][x_col], pareto_df.iloc[0][y_col]
    ax.plot([xlim_left, x0], [y0, y0],
            color=color, linestyle=linestyle, linewidth=1.8, zorder=2)

    for i in range(len(pareto_df) - 1):
        x_curr, y_curr = pareto_df.iloc[i][x_col], pareto_df.iloc[i][y_col]
        x_next, y_next = pareto_df.iloc[i + 1][x_col], pareto_df.iloc[i + 1][y_col]
        ax.plot([x_curr, x_curr], [y_curr, y_next],
                color=color, linestyle=linestyle, linewidth=1.8, zorder=2)
        ax.plot([x_curr, x_next], [y_next, y_next],
                color=color, linestyle=linestyle, linewidth=1.8, zorder=2)

    x_last, y_last = pareto_df.iloc[-1][x_col], pareto_df.iloc[-1][y_col]
    ax.plot([x_last, x_last], [y_last, ylim_left],
            color=color, linestyle=linestyle, linewidth=1.8, zorder=2)


# =========================
# 3. tool functions
# =========================
def split_sort_key(val):
    try:
        return (0, float(val))
    except Exception:
        return (1, str(val))


def build_legend_handles():
    handles = []
    for method in method_order:
        color = method_color_map[method]
        marker = method_marker_map[method]
        linestyle = METHOD_LINESTYLE.get(method, 'None')

        h = Line2D(
            [0], [0],
            color=color,
            linestyle=linestyle,
            marker=marker,
            markersize=10,
            linewidth=1.8,
            markerfacecolor=color,
            markeredgecolor=color,
            label=method
        )
        handles.append(h)
    return handles


def draw_one_panel(ax, panel_df, mode):
    x_col = f"{mode}_prec"
    y_col = f"{mode}_acc"

    ax.set_xlim(xlim_left, xlim_right + 0.03)
    ax.set_ylim(ylim_left, ylim_right + 0.03)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True)

    ax.set_xticks(tick_values)
    ax.set_yticks(tick_values)

    # scatters
    for method in method_order:
        sub = panel_df[panel_df["method"] == method]
        if sub.empty:
            continue

        sub_xy = sub[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
        if sub_xy.empty:
            continue

        ax.scatter(
            sub_xy[x_col], sub_xy[y_col],
            color=method_color_map.get(method, None),
            marker=method_marker_map.get(method, 'o'),
            s=method_size_map.get(method, 70),
            linewidths=1,
            zorder=5
        )
    # pareto cutve
    for m in methods_pareto:
        sub = panel_df[panel_df["method"] == m].copy()
        if sub.empty:
            continue

        pareto_df = pareto_frontier_max(sub, x=x_col, y=y_col)
        pareto_color = method_color_map.get(m, "black")
        pareto_linestyle = METHOD_LINESTYLE.get(m, "-")

        plot_staircase_pareto(
            ax, pareto_df, x_col, y_col,
            xlim_left, xlim_right, ylim_left, ylim_right,
            pareto_color, pareto_linestyle
        )

    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontname("Times New Roman")


# =========================
# 4. Main programming: 2\times 4 figures 
# =========================
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(csv_path).copy()
df["method"] = df["method"].astype(str).str.strip()

group_keys = ["dataset", "depth"]

for (dataset, depth), depth_df in df.groupby(group_keys):
    split_values = sorted(depth_df["split"].dropna().unique().tolist(), key=split_sort_key)
    split_values = split_values[:n_cols]

    fig = plt.figure(figsize=(22, 11.5))
    gs = fig.add_gridspec(
        nrows=3, ncols=n_cols,
        height_ratios=[0.22, 1, 1],   # legend
        hspace=0.28, wspace=0.25
    )

    legend_ax = fig.add_subplot(gs[0, :])
    legend_ax.axis("off")

    # 2\times 4 subfigures
    axes = np.empty((n_rows, n_cols), dtype=object)
    for row in range(n_rows):
        for col in range(n_cols):
            axes[row, col] = fig.add_subplot(gs[row + 1, col])

    for col in range(n_cols):
        if col < len(split_values):
            split = split_values[col]
            split_df = depth_df[depth_df["split"] == split].copy()

            for row, mode in enumerate(modes):
                ax = axes[row, col]
                draw_one_panel(ax, split_df, mode)

                if row == 0:
                    ax.set_title(f"Fold {split}", fontsize=24, fontname="Times New Roman", pad =18)

                ax.set_xlabel("Precision", fontsize=20, fontname="Times New Roman")
                ax.set_ylabel("Accuracy", fontsize=20, fontname="Times New Roman")
        else:
            for row in range(n_rows):
                axes[row, col].axis("off")

    # row
    train_y0 = min(axes[0, col].get_position().y0 for col in range(n_cols))
    train_y1 = max(axes[0, col].get_position().y1 for col in range(n_cols))
    train_y_center = 0.5 * (train_y0 + train_y1)

    test_y0 = min(axes[1, col].get_position().y0 for col in range(n_cols))
    test_y1 = max(axes[1, col].get_position().y1 for col in range(n_cols))
    test_y_center = 0.5 * (test_y0 + test_y1)

    fig.text(0.08, train_y_center, "Train",
             rotation=90, va='center', ha='center',
             fontsize=24, fontname="Times New Roman")

    fig.text(0.08, test_y_center, "Test",
             rotation=90, va='center', ha='center',
             fontsize=24, fontname="Times New Roman")

    # Horizontal legend
    legend_handles = build_legend_handles()
    legend_labels = [h.get_label() for h in legend_handles]

    legend_ax.legend(
        handles=legend_handles,
        labels=legend_labels,
        loc="center",
        ncol=len(legend_labels),
        frameon=True,
        edgecolor="black",
        prop={"family": "Times New Roman", "size": 22}
    )

    out_path = os.path.join(out_dir, f"{dataset}_depth{depth}.png")
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=SAVE_PAD_INCHES)
    plt.close(fig)

print(f"Done. Figures saved to: {out_dir}")

Done. Figures saved to: tree_pareto_curve
